In [2]:
# ==========================================
# Step 1: Import Required Libraries
# ==========================================
import pandas as pd
import numpy as np

# ==========================================
# Step 2: Load the Dataset
# ==========================================
df = pd.read_csv(r"C:\Users\thota\Downloads\reference file.csv")

# ==========================================
# Step 3: Display Basic Information
# ==========================================
print("Shape of Dataset:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
for col in df.columns:
    print(col)

Shape of Dataset: (10000, 55)

First 5 Rows:


,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,Staff Count,...,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%,Bed_Occupancy_Rate_Calc,ICU_Occupancy_Rate_Calc,Staff_Utilization_Calc,Readmission_Flag,Transferred_Flag,Equipment_InUse_Flag
0,HOSP00016,C K Hospital,Private,Mumbai,Maharashtra,General Medicine,DEPT004,Dr. Arjun Sharma,12,257,...,598,0.0,43.0,0.2,0.2,80.6,43.0,0,0,0
1,HOSP00019,Cure Well Hospital,Private,Bengaluru,Karnataka,General Surgery,DEPT010,Dr. Sneha Rao,15,288,...,583,0.2,49.4,0.2,0.2,41.0,49.4,0,0,0
2,HOSP00004,Yashoda Hospital,Private,Hyderabad,Telangana,Pulmonology,DEPT005,Dr. Priya Reddy,20,181,...,590,0.0,30.7,0.2,0.2,23.7,30.7,0,0,1
3,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Pulmonology,DEPT005,Dr. Rahul Verma,23,170,...,597,0.2,28.5,0.4,0.4,67.7,28.5,0,1,0
4,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Icu,DEPT009,Dr. Priya Reddy,34,116,...,576,0.0,20.1,0.6,0.6,39.0,20.1,0,0,1



Column Names:
Hospital ID
Hospital Name
Hospital Type
City
State
Department
Department ID
Doctor
Nurses
Staff Count
Patient ID
Patient Name
Gender
Age
Admission Date
Discharge Date
Diagnosis
Treatment
Medication
Admission Type
Test Result
Blood Type
Beds Available
ICU Beds
Bed Number
Bed Occupied
Equipment
Length of Stay
Room No
Billing Amount
Insurance Provider
Readmission
Dept_Bed_Capacity_Derived
Beds_Occupied_Count
Equipment ID
Equipment Number
Equipment Status
Equipment_Total_Inventory
Equipment_Usage_Duration_Hours
Transferred
Transfer_From_Department
Transfer_To_Department
Transfer_Date
Number_of_Transfers
Dept_ICU_Bed_Capacity_Derived
Dept_Staff_Capacity_Derived
Admissions_Rate_%_Derived
Staff_Utilization_%_Derived
Bed_Occupancy_Rate_%
Bed_Occupancy_Rate_Calc
ICU_Occupancy_Rate_Calc
Staff_Utilization_Calc
Readmission_Flag
Transferred_Flag
Equipment_InUse_Flag


In [3]:
# ==========================================
# Check Missing Values
# ==========================================
missing_values = df.isnull().sum()

# Display only columns with missing values
missing_values = missing_values[missing_values > 0]

print("Missing Values:")
print(missing_values.sort_values(ascending=False))


Missing Values:
Transfer_Date    8240
dtype: int64


In [4]:
# ==========================================
# Convert Transfer_Date to datetime
# ==========================================

df["Transfer_Date"] = pd.to_datetime(df["Transfer_Date"], errors="coerce")

print(df["Transfer_Date"].dtype)

datetime64[ns]


In [5]:
# ==========================================
# Check Duplicate Rows
# ==========================================

duplicates = df.duplicated().sum()

print("Number of Duplicate Rows:", duplicates)

Number of Duplicate Rows: 0


In [6]:
# ==========================================
# Check Data Types
# ==========================================

print(df.dtypes)

Hospital ID                               object
Hospital Name                             object
Hospital Type                             object
City                                      object
State                                     object
Department                                object
Department ID                             object
Doctor                                    object
Nurses                                     int64
Staff Count                                int64
Patient ID                                object
Patient Name                              object
Gender                                    object
Age                                        int64
Admission Date                            object
Discharge Date                            object
Diagnosis                                 object
Treatment                                 object
Medication                                object
Admission Type                            object
Test Result         

In [8]:
flag_columns = [
    "Readmission_Flag",
    "Transferred_Flag",
    "Equipment_InUse_Flag"
]

for col in flag_columns:
    df[col] = df[col].astype(bool)

print(df[flag_columns].dtypes)

Readmission_Flag        bool
Transferred_Flag        bool
Equipment_InUse_Flag    bool
dtype: object


In [9]:
# ==========================================
# Convert Boolean Flags Back to 0/1
# ==========================================

flag_columns = [
    "Readmission_Flag",
    "Transferred_Flag",
    "Equipment_InUse_Flag"
]

for col in flag_columns:
    df[col] = df[col].astype(int)

print(df[flag_columns].dtypes)
print(df[flag_columns].head())

Readmission_Flag        int64
Transferred_Flag        int64
Equipment_InUse_Flag    int64
dtype: object
   Readmission_Flag  Transferred_Flag  Equipment_InUse_Flag
0                 0                 0                     0
1                 0                 0                     0
2                 0                 0                     1
3                 0                 1                     0
4                 0                 0                     1


In [10]:
# ==========================================
# Convert Date Columns
# ==========================================

date_columns = [
    "Admission Date",
    "Discharge Date",
    "Transfer_Date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

Admission Date    datetime64[ns]
Discharge Date    datetime64[ns]
Transfer_Date     datetime64[ns]
dtype: object


In [11]:
# ==========================================
# Validate Length of Stay
# ==========================================

df["Calculated_Length_of_Stay"] = (
    df["Discharge Date"] - df["Admission Date"]
).dt.days

los_mismatch = df[
    df["Length of Stay"] != df["Calculated_Length_of_Stay"]
]

print("Number of mismatches:", len(los_mismatch))

display(
    los_mismatch[
        [
            "Patient ID",
            "Admission Date",
            "Discharge Date",
            "Length of Stay",
            "Calculated_Length_of_Stay"
        ]
    ].head()
)

Number of mismatches: 6051


,Patient ID,Admission Date,Discharge Date,Length of Stay,Calculated_Length_of_Stay
2,PAT00003,2026-01-04,2026-05-04,4,120
4,PAT00005,2025-09-11,2025-12-11,3,91
5,PAT00006,2025-02-21,2025-06-03,13,102
6,PAT00007,2025-01-21,2025-03-02,13,40
11,PAT00012,2026-05-27,2026-01-06,5,-141


In [12]:
los_mismatch[
    [
        "Patient ID",
        "Admission Date",
        "Discharge Date",
        "Length of Stay",
        "Calculated_Length_of_Stay"
    ]
].head(20)

,Patient ID,Admission Date,Discharge Date,Length of Stay,Calculated_Length_of_Stay
2,PAT00003,2026-01-04,2026-05-04,4,120
4,PAT00005,2025-09-11,2025-12-11,3,91
5,PAT00006,2025-02-21,2025-06-03,13,102
6,PAT00007,2025-01-21,2025-03-02,13,40
11,PAT00012,2026-05-27,2026-01-06,5,-141
14,PAT00015,2025-01-29,2025-12-02,14,307
17,PAT00018,2025-10-29,2025-04-11,6,-201
18,PAT00019,2025-08-06,2025-06-16,8,-51
19,PAT00020,2025-09-07,2025-11-07,2,61
22,PAT00023,2026-01-27,2026-07-02,11,156


In [13]:
df[["Admission Date", "Discharge Date"]].head(10)

,Admission Date,Discharge Date
0,2026-02-13,2026-02-27
1,2026-05-16,2026-05-20
2,2026-01-04,2026-05-04
3,2025-09-13,2025-09-24
4,2025-09-11,2025-12-11
5,2025-02-21,2025-06-03
6,2025-01-21,2025-03-02
7,2026-05-13,2026-05-24
8,2025-01-21,2025-01-31
9,2025-04-17,2025-04-27


In [14]:
# ==========================================
# Check for invalid dates
# ==========================================

invalid_dates = df[df["Discharge Date"] < df["Admission Date"]]

print("Number of records with discharge before admission:", len(invalid_dates))

display(
    invalid_dates[
        [
            "Patient ID",
            "Admission Date",
            "Discharge Date",
            "Length of Stay"
        ]
    ].head(10)
)

Number of records with discharge before admission: 2964


,Patient ID,Admission Date,Discharge Date,Length of Stay
11,PAT00012,2026-05-27,2026-01-06,5
17,PAT00018,2025-10-29,2025-04-11,6
18,PAT00019,2025-08-06,2025-06-16,8
23,PAT00024,2026-10-03,2026-03-18,8
25,PAT00026,2025-07-01,2025-01-21,14
27,PAT00028,2025-12-08,2025-08-14,2
28,PAT00029,2026-12-04,2026-04-18,6
31,PAT00032,2025-05-28,2025-04-06,7
33,PAT00034,2025-11-06,2025-06-15,4
35,PAT00036,2025-09-04,2025-04-13,4


In [15]:
# ==========================================
# Swap Admission and Discharge Dates
# where they are reversed
# ==========================================

mask = df["Discharge Date"] < df["Admission Date"]

df.loc[mask, ["Admission Date", "Discharge Date"]] = (
    df.loc[mask, ["Discharge Date", "Admission Date"]].values
)

print("Dates corrected.")

Dates corrected.


In [16]:
# ==========================================
# Recalculate Length of Stay
# ==========================================

df["Length of Stay"] = (
    df["Discharge Date"] - df["Admission Date"]
).dt.days

print("Length of Stay recalculated.")

Length of Stay recalculated.


In [17]:
df["Calculated_Length_of_Stay"] = (
    df["Discharge Date"] - df["Admission Date"]
).dt.days

mismatch = df[df["Length of Stay"] != df["Calculated_Length_of_Stay"]]

print("Length of Stay mismatches:", len(mismatch))

Length of Stay mismatches: 0


In [18]:
#Validate Transferred vs Transferred_Flag
transfer_check = df[
    ((df["Transferred"] == "Yes") & (df["Transferred_Flag"] != 1)) |
    ((df["Transferred"] == "No") & (df["Transferred_Flag"] != 0))
]

print("Transfer mismatches:", len(transfer_check))

Transfer mismatches: 0


In [19]:
#Validate Readmission vs Readmission_Flag
readmission_check = df[
    ((df["Readmission"] == "Yes") & (df["Readmission_Flag"] != 1)) |
    ((df["Readmission"] == "No") & (df["Readmission_Flag"] != 0))
]

print("Readmission mismatches:", len(readmission_check))

Readmission mismatches: 0


In [20]:
#Validate Equipment Status vs Equipment_InUse_Flag
print(df["Equipment Status"].unique())

['Under Maintenance' 'In Use' 'Available']


In [21]:
# ==========================================
# Validate Equipment Status vs Equipment_InUse_Flag
# ==========================================

equipment_mismatch = df[
    ((df["Equipment Status"] == "In Use") & (df["Equipment_InUse_Flag"] != 1)) |
    ((df["Equipment Status"] == "Available") & (df["Equipment_InUse_Flag"] != 0)) |
    ((df["Equipment Status"] == "Under Maintenance") & (df["Equipment_InUse_Flag"] != 0))
]

print("Equipment Status mismatches:", len(equipment_mismatch))

display(
    equipment_mismatch[
        ["Equipment ID", "Equipment Status", "Equipment_InUse_Flag"]
    ].head(10)
)

Equipment Status mismatches: 0


,Equipment ID,Equipment Status,Equipment_InUse_Flag


In [22]:
# ==========================================
# Validate Bed Occupancy Rate
# ==========================================
bed_mismatch = df[
    round(df["Bed_Occupancy_Rate_%"], 2) !=
    round(df["Bed_Occupancy_Rate_Calc"], 2)
]

print("Bed Occupancy mismatches:", len(bed_mismatch))

Bed Occupancy mismatches: 0


In [23]:
# ==========================================
# Validate Staff Utilization
# ========================================
staff_mismatch = df[
    round(df["Staff_Utilization_%_Derived"], 2) !=
    round(df["Staff_Utilization_Calc"], 2)
]

print("Staff Utilization mismatches:", len(staff_mismatch))

Staff Utilization mismatches: 0


In [24]:
print(df[[
    "ICU Beds",
    "Dept_ICU_Bed_Capacity_Derived",
    "ICU_Occupancy_Rate_Calc"
]].head(10))

   ICU Beds  Dept_ICU_Bed_Capacity_Derived  ICU_Occupancy_Rate_Calc
0        79                             98                     80.6
1        41                            100                     41.0
2        23                             97                     23.7
3        67                             99                     67.7
4        39                            100                     39.0
5        50                             86                     58.1
6        82                             99                     82.8
7        17                             96                     17.7
8        36                            100                     36.0
9        39                            100                     39.0


In [25]:
# ==========================================
# Validate ICU Occupancy Rate
# ==========================================

df["Calculated_ICU_Occupancy"] = round(
    (df["ICU Beds"] / df["Dept_ICU_Bed_Capacity_Derived"]) * 100,
    1
)

icu_mismatch = df[
    df["Calculated_ICU_Occupancy"] != df["ICU_Occupancy_Rate_Calc"]
]

print("ICU Occupancy mismatches:", len(icu_mismatch))

display(
    icu_mismatch[
        [
            "ICU Beds",
            "Dept_ICU_Bed_Capacity_Derived",
            "Calculated_ICU_Occupancy",
            "ICU_Occupancy_Rate_Calc"
        ]
    ].head(10)
)


ICU Occupancy mismatches: 0


,ICU Beds,Dept_ICU_Bed_Capacity_Derived,Calculated_ICU_Occupancy,ICU_Occupancy_Rate_Calc


In [26]:
# ==========================================
# Summary Statistics for Numerical Columns
# ==========================================

df.describe().T

,count,mean,min,25%,50%,75%,max,std
Nurses,10000.0,27.5361,5.0,16.0,28.0,39.0,50.0,13.24947
Staff Count,10000.0,341.421,80.0,214.75,343.0,470.0,600.0,149.201603
Age,10000.0,45.3039,1.0,23.0,45.0,68.0,90.0,25.952659
Admission Date,10000,2025-09-01 06:48:05.760000,2025-01-01 00:00:00,2025-04-05 00:00:00,2025-08-09 00:00:00,2026-02-01 06:00:00,2026-11-06 00:00:00,NaN
Discharge Date,10000,2025-11-19 01:34:01.920000,2025-01-13 00:00:00,2025-07-16 00:00:00,2025-10-31 00:00:00,2026-03-30 00:00:00,2026-12-06 00:00:00,NaN
Beds Available,10000.0,258.8933,20.0,137.0,258.0,381.0,500.0,139.770105
ICU Beds,10000.0,52.6759,5.0,29.0,53.0,76.0,100.0,27.745581
Bed Number,10000.0,251.1153,1.0,128.0,250.0,377.0,500.0,143.325642
Length of Stay,10000.0,78.7819,1.0,8.0,44.0,134.0,338.0,84.444638
Room No,10000.0,550.3256,100.0,326.0,550.0,776.25,999.0,258.519784


In [27]:
df.drop(columns=[
    "Calculated_Length_of_Stay",
    "Calculated_ICU_Occupancy"
], inplace=True)

In [28]:
print((df["Beds Available"] < 0).sum())
print((df["ICU Beds"] < 0).sum())
print((df["Billing Amount"] < 0).sum())
print((df["Age"] < 0).sum())

0
0
0
0


In [29]:
# ==========================================
# Save Cleaned Dataset
# ==========================================

df.to_csv("hospital_cleaned.csv", index=False)

print("Dataset saved successfully as 'hospital_cleaned.csv'")

Dataset saved successfully as 'hospital_cleaned.csv'
